# Hybrid Recommender — EDA & Model Exploration

This notebook walks through:
1. Dataset statistics
2. Rating distributions
3. TF-IDF vocabulary inspection
4. SVD latent space visualisation
5. Hybrid score analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

sns.set_theme(style='dark', palette='muted')
plt.rcParams['figure.facecolor'] = '#0a0a0f'
plt.rcParams['axes.facecolor']   = '#13131a'
plt.rcParams['text.color']       = '#e8e6f0'
plt.rcParams['axes.labelcolor']  = '#e8e6f0'
plt.rcParams['xtick.color']      = '#6b6880'
plt.rcParams['ytick.color']      = '#6b6880'

DATA_DIR   = Path('../data')
MODELS_DIR = Path('../models')

## 1. Dataset Overview

In [ ]:
movies = pd.read_csv(DATA_DIR / 'movies.csv')
books  = pd.read_csv(DATA_DIR / 'books.csv')

print('── Movies ──')
print(f'  Rows    : {len(movies):,}')
print(f'  Avg rating: {movies.avg_rating.mean():.2f}')
print(f'  Columns : {list(movies.columns)}')

print('\n── Books ──')
print(f'  Rows    : {len(books):,}')
print(f'  Avg rating: {books.avg_rating.mean():.2f}')
print(f'  Columns : {list(books.columns)}')

## 2. Rating Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(movies['avg_rating'], bins=40, color='#7c6af0', alpha=0.8, edgecolor='none')
axes[0].set_title('Movie Rating Distribution', color='#e8e6f0')
axes[0].set_xlabel('Average Rating')

axes[1].hist(books['avg_rating'], bins=40, color='#c8a96e', alpha=0.8, edgecolor='none')
axes[1].set_title('Book Rating Distribution', color='#e8e6f0')
axes[1].set_xlabel('Average Rating')

plt.tight_layout()
plt.show()

## 3. Genre Analysis (Movies)

In [ ]:
from collections import Counter

all_genres = []
for genres in movies['genres_clean'].dropna():
    all_genres.extend(genres.split())

genre_counts = Counter(all_genres).most_common(15)
labels, vals = zip(*genre_counts)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(labels[::-1], vals[::-1], color='#7c6af0', alpha=0.85)
ax.set_title('Top 15 Movie Genres', color='#e8e6f0')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

## 4. TF-IDF Top Terms

In [ ]:
bundle = joblib.load(MODELS_DIR / 'movie_model.joblib')
tfidf  = bundle['tfidf']

vocab  = tfidf.vocabulary_
idf    = tfidf.idf_
sorted_terms = sorted(vocab.items(), key=lambda x: idf[x[1]])

print('Lowest IDF (most common) terms:')
for term, idx in sorted_terms[:20]:
    print(f'  {term:<25} idf={idf[idx]:.3f}')

print('\nHighest IDF (rarest) terms:')
for term, idx in sorted_terms[-10:]:
    print(f'  {term:<25} idf={idf[idx]:.3f}')

## 5. Live Recommendation Test

In [ ]:
import sys; sys.path.insert(0, '..')
from api.recommender import recommender

query = 'space exploration alien planets'

print(f'Query: "{query}"\n')
print('── Top Movies ──')
for r in recommender.recommend(query, 'movie', top_n=5):
    print(f"  {r['title']:<45} hybrid={r['hybrid_score']:.3f}  content={r['content_score']:.3f}")

print('\n── Top Books ──')
for r in recommender.recommend(query, 'book', top_n=5):
    print(f"  {r['title']:<45} hybrid={r['hybrid_score']:.3f}  content={r['content_score']:.3f}")

## 6. Score Correlation

In [ ]:
results = recommender.recommend('thriller mystery crime', 'movie', top_n=50)
df_res  = pd.DataFrame(results)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(
    df_res['content_score'], df_res['collab_score'],
    c=df_res['hybrid_score'], cmap='plasma',
    s=80, alpha=0.8
)
plt.colorbar(sc, ax=ax, label='Hybrid Score')
ax.set_xlabel('Content Score (TF-IDF)')
ax.set_ylabel('Collab / Popularity Score')
ax.set_title('Content vs Collaborative Scores', color='#e8e6f0')
plt.tight_layout()
plt.show()